# Layer 1: vendor-level baseline
Layer 1 only uses information about one vendor at a time:
- the vendor's own fields
- its own rows in the sub-tables (engagements, payments, approvals, services, institution links, addresses)
- reference tables (approval policy, employee roles, country risk)

Check: adding a new vendor shouldn't change any existing vendor's values. "Number of sponsors of this vendor" is Layer 1, "which vendors share a sponsor" is Layer 2.

Outline:
1. load, clean, build y and features
2. supervised: variable selection with 4 penalized models, 5 models on one 4:1 split
3. unsupervised: Isolation Forest
4. out-of-fold evaluation over 5 seeds + ablations
5. save outputs for Layer 2

y = 0 means "not in a labelled case", not "clean", so precision is a lower bound.

## Setup and loading
`DATA_DIR`: folder with the csv files. `N_SEEDS` for the 5-seed evaluation, `N_JOBS` for parallel folds.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import re
import os
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "student_bundle"
N_SEEDS = 5
N_JOBS = 2
SNAPSHOT = pd.Timestamp("2026-06-30")
ERP_DATE = pd.Timestamp("2022-05-01")          # ERP migration (business_events)
GENERIC_EMAIL = {"gmail.com", "outlook.com", "yahoo.com", "hotmail.com", "icloud.com"}


def rd(name, dates=()):
    df = pd.read_csv(os.path.join(DATA_DIR, name))
    for c in dates:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


tp = rd("third_parties.csv", ["onboarding_date", "created_date", "terminated_date"])
dd = rd("tp_due_diligence.csv", ["dd_completed", "dd_refresh_due"])
addr = rd("tp_address_history.csv", ["valid_from", "valid_to", "observed_at"])
svc = rd("tp_services.csv", ["valid_from", "valid_to"])
eng = rd("tp_engagements.csv", ["start_date", "end_date", "observed_at"])
appr = rd("tp_engagement_approvals.csv", ["approval_date", "observed_at"])
pay = rd("tp_payments_monthly.csv", ["month"])
inst_l = rd("tp_institution_links.csv", ["valid_from", "valid_to"])
inst = rd("healthcare_institutions.csv")
roles = rd("employee_roles.csv", ["valid_from", "valid_to"])
cats = rd("service_categories.csv").rename(columns={"flags": "rflags"})
policy = rd("approval_policy_matrix.csv")
crc = rd("country_risk_context.csv", ["as_of_date"])
mkts = rd("markets.csv")
fmv = rd("fmv_rate_cards.csv", ["effective_from", "effective_to"])
labels = rd("labelled_cases_TRAINING_SET.csv")
OUT = 'results'
os.makedirs(OUT, exist_ok=True)

from IPython.display import display
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
print('third_parties:', tp.shape, '| engagements:', eng.shape, '| payments:', pay.shape)

## Cleaning
- strip strings, empty string → NaN, drop exact duplicate rows
- null `valid_to` / `end_date` = still open → filled with the snapshot date (`*_f` columns)
- service categories come in two taxonomy versions; map v1 → v2 with the crosswalk and take the union of the risk flags
- the `system_default` sponsor is the ERP-migration fallback account, not a person → NaN in `sponsor_clean`
- addresses: lower-case, drop punctuation, expand abbreviations, sort tokens, merge small typos. Typos are only merged within one vendor's own addresses; merging across vendors would be Layer 2.

In [ ]:
# strip strings, "" -> NaN
for df in [tp, dd, addr, svc, eng, appr, pay, inst_l, roles, cats]:
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().replace({"": pd.NA})

# exact duplicate rows
for name, df in [("addr", addr), ("svc", svc), ("eng", eng), ("appr", appr), ("pay", pay)]:
    n0 = len(df)
    df.drop_duplicates(inplace=True)
    if n0 != len(df):
        print(f"[clean] {name}: dropped {n0 - len(df)} exact duplicate rows")

# null valid_to / end_date = still open
addr["valid_to_f"] = addr["valid_to"].fillna(SNAPSHOT)
svc["valid_to_f"] = svc["valid_to"].fillna(SNAPSHOT)
eng["end_f"] = eng["end_date"].fillna(SNAPSHOT)
inst_l["valid_to_f"] = inst_l["valid_to"].fillna(SNAPSHOT)

# taxonomy v1 -> v2: a v1 code can map to several v2 codes, take the union of flags
cats["rflags"] = cats["rflags"].fillna("")
flag_v2 = dict(zip(cats.category_code_v2, cats["rflags"]))
parent_v2 = dict(zip(cats.category_code_v2, cats.parent))
flag_v1 = cats.groupby("category_code_v1")["rflags"].apply(
    lambda s: ";".join(sorted({f for x in s for f in x.split(";") if f}))).to_dict()
parent_v1 = cats.groupby("category_code_v1")["parent"].first().to_dict()


def cat_flags(code, ver):
    return (flag_v2 if ver == "v2" else flag_v1).get(code, "")


def cat_parent(code, ver):
    p = (parent_v2 if ver == "v2" else parent_v1).get(code)
    return p if p is not None else (code.split(".")[0] if isinstance(code, str) else None)


eng["rflags"] = [cat_flags(c, v) for c, v in zip(eng.service_category, eng.taxonomy_version)]
eng["cat_parent"] = [cat_parent(c, v) for c, v in zip(eng.service_category, eng.taxonomy_version)]
svc["rflags"] = [cat_flags(c, v) for c, v in zip(svc.category_code, svc.taxonomy_version)]

# system_default sponsor = ERP fallback account
role_of = dict(zip(roles.employee_id, roles.role))
lvl_of = dict(zip(roles.employee_id, roles.approver_level))
sys_emps = set(roles.loc[roles.role == "system_default", "employee_id"])
eng["sponsor_is_system"] = eng.sponsor_employee_id.isin(sys_emps)
eng["sponsor_clean"] = eng.sponsor_employee_id.where(~eng.sponsor_is_system)  # NaN if system

# address normalisation
ABBR = {"st": "street", "str": "street", "rd": "road", "ave": "avenue", "av": "avenue",
        "blvd": "boulevard", "ste": "suite", "fl": "floor", "apt": "apartment", "dr": "drive",
        "ln": "lane", "ct": "court", "hwy": "highway", "no": "number", "bldg": "building",
        "sq": "square", "pl": "place", "ctr": "center", "centre": "center"}


def norm_addr(s):
    if pd.isna(s):
        return None
    s = str(s).lower()
    s = re.sub(r"[^\w\s]", " ", s)                  # punctuation -> space
    toks = [ABBR.get(t, t) for t in s.split()]
    return " ".join(sorted(toks))                    # sort tokens so field order doesn't matter


addr["addr_key"] = addr.raw_address.map(norm_addr) + "|" + addr.address_country.fillna("")


# merge keys in the same country that differ by one token
def merge_typos(keys):
    parent = {k: k for k in keys}

    def f(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    by_c = defaultdict(list)
    for k in keys:
        by_c[k.split("|")[1]].append(k)
    for _, ks in by_c.items():
        toks = {k: set(k.split("|")[0].split()) for k in ks}
        for a, b in combinations(ks, 2):
            ta, tb = toks[a], toks[b]
            if len(ta) >= 4 and len(ta ^ tb) <= 2 and len(ta & tb) >= len(ta) - 1:
                parent[f(a)] = f(b)
    return {k: f(k) for k in keys}


# only within one vendor's addresses
def merge_typos_within_vendor(df):
    out = df["addr_key"].copy()
    for rid, idx in df.dropna(subset=["addr_key"]).groupby("record_id").groups.items():
        keys = df.loc[idx, "addr_key"].unique()
        if len(keys) > 1:
            out.loc[idx] = df.loc[idx, "addr_key"].map(merge_typos(keys))
    return out


addr["addr_key"] = merge_typos_within_vendor(addr)

# country context (CPI is in the feature cell)
bo = crc[crc["index"] == "BO_REGISTER_ACCESS"].sort_values("as_of_date").groupby("iso2").last()
bo_map = bo["tier"].map({"high": 0, "medium": 1, "low": 2}).to_dict()
sanc = set(crc.loc[crc["index"] == "SANCTIONS", "iso2"])
region = dict(zip(mkts.market_id, mkts.region))
print('cleaning done | distinct normalized addresses:', addr.addr_key.nunique())

## y and groups
- y = 1 if the record is in any labelled case (159 of 1,002, ~16%)
- `group_id` keeps all members of a case on the same side of a split. Two records are in two cases each, so cases that share a member are merged with union-find → 31 positive groups. Each negative is its own group.

In [ ]:
labels["members"] = labels.member_record_ids.str.split("|")
pos_ids = set(m for ms in labels.members for m in ms)
# union-find over cases that share a member (TP-98B17648, TP-42DFE96B appear in 2 cases)
par = {}


def uf(x):
    par.setdefault(x, x)
    while par[x] != x:
        par[x] = par[par[x]]
        x = par[x]
    return x


for ms in labels.members:
    for m in ms[1:]:
        par[uf(m)] = uf(ms[0])
    uf(ms[0])

X = tp[["record_id"]].copy()
X["y"] = X.record_id.isin(pos_ids).astype(int)
# negatives: one group each
X["group_id"] = [("G_" + uf(r)) if r in pos_ids else ("N_" + r) for r in X.record_id]

lv = labels[['case_id', 'typology_tags', 'difficulty_tier', 'market']].copy()
lv['n_members'] = labels.members.str.len()
display(lv)
print('y:', X.y.value_counts().to_dict(), '| positive groups:', X.loc[X.y == 1, 'group_id'].nunique())

## Layer 1 features

One row per vendor (variable list with sources and definitions: `layer1_variable_list.xlsx`). Four groups:

- **A** own fields: `is_terminated`, `risk_rating_ord`, DD answers, ...
- **B** aggregates of the vendor's own sub-table rows: payment-channel shares, number of engagements, address types, ...
- **C** counts / attributes of the linked employees, institutions and markets: `n_sponsors`, `n_exceeds_authority`, `n_inst_links`, ...
- **D** country / region: `country_cpi`, `region_*`

C touches employee and institution nodes, but only through this vendor's own engagements and links. Its effect is checked in the ablations.

Notes:
- **Authority check.** For each financial approval, take the approver's role row valid on the approval date and compare `approver_level` with the level required for the engagement's `value_band` (`approval_policy_matrix`). `employee_delegations` is empty.
  - `n_exceeds_authority`: the approver had a valid role, but the level was too low
  - `n_approver_role_missing`: approver ID present, but no valid role row on that date. Kept separate from "exceeding" since it could also be a data gap. (No approver at all → `share_approver_missing`.)
- **CPI**: the edition in force at onboarding, not the latest. Vendors onboarded before the first edition (2013) get the earliest one.
- **Missing values**: counts are 0 when there are no rows; shares and means stay NaN (no payments → `share_non_po` = NaN). No extra missing indicators, the counts already cover that.
- Winsorizing and imputation are done inside the model pipeline (`Prep()`).

In [ ]:
feats = {}          # name -> pd.Series indexed by record_id
R = tp.set_index("record_id")


def add(name, s, fill=0.0):
    feats[name] = s.reindex(R.index).astype(float).fillna(fill)


# third_parties
add("is_individual", (R.party_type == "individual"))
add("is_terminated", (R.status == "terminated"))
end_ref = R.terminated_date.fillna(SNAPSHOT)
add("tenure_years", (end_ref - R.onboarding_date).dt.days / 365.25)
add("onboard_year", R.onboarding_date.dt.year, fill=2018)
add("onboarded_last_3y", R.onboarding_date >= SNAPSHOT - pd.DateOffset(years=3))
add("migrated_erp", (R.created_date == ERP_DATE) | (R.created_date - R.onboarding_date).dt.days.gt(30))
add("created_minus_onboard_days", (R.created_date - R.onboarding_date).dt.days.clip(-365, 365))
add("has_lei", R.lei.notna())
add("has_registry_no", R.registry_no.notna())
add("has_declared_parent", R.declared_parent_group.notna())
add("remit_country_differs", R.remit_to_country != R.country)
add("generic_email", R.email_domain.str.lower().isin(GENERIC_EMAIL))
for et in ["hcp_consultant", "investigator_kol", "writer_translator", "regulatory_access",
           "patient_expert", "other_consultant"]:
    add(f"expert_{et}", R.expert_type == et)
# country: CPI edition at onboarding (earliest edition if onboarded before it)
_cpi = (crc[(crc["index"] == "CPI") & crc["value"].notna()].sort_values("as_of_date")
        [["iso2", "as_of_date", "value", "tier"]].rename(columns={"iso2": "country"}))
_cpi["tier"] = pd.to_numeric(_cpi["tier"], errors="coerce")
_cpi["country"] = _cpi["country"].astype(object)
_o = tp[["record_id", "country", "onboarding_date"]].dropna().sort_values("onboarding_date")
_o["country"] = _o["country"].astype(object)
_b = pd.merge_asof(_o, _cpi, left_on="onboarding_date", right_on="as_of_date", by="country", direction="backward")
_f = pd.merge_asof(_o, _cpi, left_on="onboarding_date", right_on="as_of_date", by="country", direction="forward")
_b["value"], _b["tier"] = _b["value"].fillna(_f["value"]), _b["tier"].fillna(_f["tier"])
_b = _b.set_index("record_id")
add("country_cpi", _b["value"], fill=np.nan)
add("country_cpi_tier", _b["tier"], fill=np.nan)
add("country_bo_access_low", R.country.map(bo_map), fill=1)
add("country_sanctioned", R.country.isin(sanc))
for rg in ["US", "EUROPE", "JAPAN_CHINA", "INTL"]:
    add(f"region_{rg}", R.country.map(region) == rg)

# due diligence
D = dd.set_index("record_id")
add("dd_enhanced", D.dd_level == "enhanced")
for q in ["q_gov_ownership", "q_pep_connected", "q_public_official_interaction",
          "q_former_public_official", "adverse_media"]:
    add(q, D[q].astype(str).str.lower() == "true")
add("screening_potential_match", D.screening_result == "potential_match_cleared")
add("risk_rating_ord", D.risk_rating.map({"low": 0, "medium": 1, "high": 2}), fill=0)
add("dd_overdue", (D.dd_refresh_due < SNAPSHOT) & (R.reindex(D.index).status == "active"))
add("dd_before_onboard_days",
    (R.onboarding_date.reindex(D.index) - D.dd_completed).dt.days.clip(-365, 730))

# services
g = svc.groupby("record_id")
add("n_service_rows", g.size())
add("n_service_cats", g.category_code.nunique())
for fl in ["hcp", "gov", "int", "flow"]:
    add(f"svc_flag_{fl}", g["rflags"].apply(lambda s, fl=fl: any(fl in x.split(";") for x in s)))
add("svc_added_after_onboard",
    svc.merge(tp[["record_id", "onboarding_date"]]).assign(
        late=lambda d: (d.valid_from - d.onboarding_date).dt.days > 180
        ).groupby("record_id").late.sum())

# engagements
g = eng.groupby("record_id")
add("n_engagements", g.size())
add("n_sponsors", g.sponsor_clean.nunique())
add("share_sponsor_missing", g.sponsor_employee_id.apply(lambda s: s.isna().mean()))
add("share_sponsor_system", g.sponsor_is_system.mean())
add("n_business_units", g.business_unit.nunique())
add("n_eng_markets", g.market.nunique())
eng = eng.merge(tp[["record_id", "country"]], on="record_id", how="left")
add("share_cross_border", eng.assign(cb=eng.market != eng.country).groupby("record_id").cb.mean())
add("max_value_band", g.value_band.max())
add("mean_value_band", g.value_band.mean())
add("share_contract_none", g.contract_type.apply(lambda s: (s == "none").mean()))
add("share_eng_flag_int", g["rflags"].apply(lambda s: np.mean(["int" in x for x in s])))
add("share_eng_flag_gov", g["rflags"].apply(lambda s: np.mean(["gov" in x for x in s])))
add("eng_late_record_days", eng.assign(
    lag=(eng.observed_at - eng.start_date).dt.days).groupby("record_id").lag.max().clip(0, 365))

# approvals: authority + compliance review
ap = appr.merge(eng[["engagement_id", "record_id", "value_band", "rflags", "sponsor_clean",
                     "start_date"]], on="engagement_id", how="left")
req = dict(zip(policy.value_band, policy.required_approver_level))
# approver level from the role row valid on the approval date
_rl = roles.set_index("employee_id")
_lv = ap.approver_employee_id.map(_rl.approver_level)
_ok = ((ap.approval_date >= ap.approver_employee_id.map(_rl.valid_from)) &
       (ap.approval_date <= ap.approver_employee_id.map(_rl.valid_to.fillna(SNAPSHOT))))
ap["approver_level"] = np.where(_ok, _lv, np.nan)        # no valid role -> NaN
ap["role_missing"] = ap.approver_employee_id.notna() & ~_ok
print("financial approvals whose approver had no valid role on the approval date:",
      int(((~_ok) & _lv.notna() & (ap.approval_type == "financial")).sum()))
ap["required_level"] = ap.value_band.map(req)
fin = ap[ap.approval_type == "financial"].copy()
fin["exceeds_authority"] = fin.approver_level < fin.required_level
fin["approver_is_sponsor"] = fin.approver_employee_id == fin.sponsor_clean
fin["approved_after_start"] = fin.approval_date > fin.start_date
g = fin.groupby("record_id")
add("n_fin_approvals", g.size())
add("n_exceeds_authority", g.exceeds_authority.sum())
add("n_approver_role_missing", g.role_missing.sum())
add("share_approver_missing", g.approver_employee_id.apply(lambda s: s.isna().mean()))
add("n_approver_is_sponsor", g.approver_is_sponsor.sum())
add("share_approved_after_start", g.approved_after_start.mean())
add("n_distinct_approvers", g.approver_employee_id.nunique())
need_cr = eng[eng["rflags"].str.contains("hcp|gov|int", regex=True)][["engagement_id", "record_id"]]
has_cr = set(appr.loc[appr.approval_type == "compliance_review", "engagement_id"])
need_cr = need_cr.assign(missing=~need_cr.engagement_id.isin(has_cr))
add("n_missing_compliance_review", need_cr.groupby("record_id").missing.sum())

# payments
pay = pay.merge(tp[["record_id", "terminated_date", "expert_type", "country"]], on="record_id",
                how="left")
g = pay.groupby("record_id")
tot = g.amount_usd.sum()
add("log_total_spend", np.log1p(tot.clip(lower=0)))
add("n_pay_months", g.month.nunique())
add("share_non_po", pay.assign(w=pay.amount_usd * (pay.channel != "po_backed")).groupby(
    "record_id").w.sum() / tot)
add("share_pcard_expense", pay.assign(w=pay.amount_usd * pay.channel.isin(
    ["p_card", "expense_reimbursement"])).groupby("record_id").w.sum() / tot)
add("manual_pay_rate", g.manual_payment_count.sum() / g.payment_count.sum().replace(0, np.nan))
add("mean_round_share", g.round_amount_share.mean())
add("n_expense_classes", g.expense_classification.nunique())
add("share_tov", pay.assign(w=pay.amount_usd * (pay.tov_category != "not_applicable")).groupby(
    "record_id").w.sum() / tot)
add("share_paying_mkt_differs", pay.assign(w=pay.amount_usd * (pay.paying_market != pay.vendor_country)
                                            ).groupby("record_id").w.sum() / tot)
add("share_commission_cash", pay.assign(w=pay.amount_usd * pay.expense_classification.isin(
    ["commissions", "grants_donations", "government_fees", "sponsorships"])).groupby(
    "record_id").w.sum() / tot)
# invoice larger than the contract band
pb = pay.merge(eng[["engagement_id", "value_band"]], on="engagement_id", how="left")
add("n_invoice_above_band", pb.assign(x=pb.max_invoice_band > pb.value_band).groupby("record_id").x.sum())
# payments after termination
add("n_months_paid_after_term", pay.assign(
    x=pay.terminated_date.notna() & (pay.month > pay.terminated_date)).groupby("record_id").x.sum())
# monthly spend shape
monthly = pay.groupby(["record_id", "month"]).amount_usd.sum().reset_index()
mg = monthly.groupby("record_id").amount_usd
add("spike_ratio", mg.max() / mg.median().replace(0, np.nan), fill=1)
add("top3_month_share", mg.apply(lambda s: s.nlargest(3).sum() / max(s.sum(), 1e-9)))
add("cv_monthly_spend", mg.std() / mg.mean().replace(0, np.nan))

# FMV: annual spend vs cap, individual experts
def band(c):
    r = region.get(c)
    if c == "US":
        return "US"
    if r == "EUROPE" or c == "JP":
        return "EU_JP"
    return "EMERGING"


cap = fmv.groupby(["card_version", "country_band", "expert_role"]).annual_cap_usd.max()
ind = pay[pay.expert_type.notna()].copy()
ind["year"] = ind.month.dt.year
ann = ind.groupby(["record_id", "year", "expert_type", "country"]).amount_usd.sum().reset_index()
ann["ver"] = np.where(ann.year <= 2023, "v1", "v2")
ann["cap"] = [cap.get((v, band(c), e), np.nan) for v, c, e in zip(ann.ver, ann.country, ann.expert_type)]
add("max_fmv_cap_ratio", (ann.amount_usd / ann.cap).groupby(ann.record_id).max())

# institution links
il = inst_l.merge(inst, on="institution_id", how="left").merge(
    tp[["record_id", "country"]].rename(columns={"country": "tp_country"}), on="record_id")
g = il.groupby("record_id")
add("n_inst_links", g.size())
add("n_inst_countries", g.country.nunique())
add("n_links_foreign", il.assign(x=il.country != il.tp_country).groupby("record_id").x.sum())
add("share_links_public", g.public_sector.mean())
add("n_link_serves", il.assign(x=il.link_type == "SERVES").groupby("record_id").x.sum())

# addresses
g = addr.groupby("record_id")
add("n_addresses", g.addr_key.nunique())
for t in ["virtual_office", "registered_agent", "po_box", "residential", "coworking"]:
    add(f"addr_{t}", g.address_type.apply(lambda s, t=t: (s == t).any()))
add("share_addr_type_missing", g.address_type.apply(lambda s: s.isna().mean()))
add("addr_country_differs", addr.merge(tp[["record_id", "country"]]).assign(
    x=lambda d: d.address_country != d.country).groupby("record_id").x.any())
add("addr_changed_after_onboard", addr.merge(tp[["record_id", "onboarding_date"]]).assign(
    x=lambda d: (d.valid_from - d.onboarding_date).dt.days > 30).groupby("record_id").x.any())

# assemble
F = pd.DataFrame(feats).replace([np.inf, -np.inf], np.nan)
# shares / means are NaN when there are no underlying rows
undefined_if_zero = {
    "n_engagements": ["share_sponsor_missing", "share_sponsor_system", "share_cross_border", "max_value_band",
                      "mean_value_band", "share_contract_none", "share_eng_flag_int", "share_eng_flag_gov",
                      "eng_late_record_days"],
    "n_fin_approvals": ["share_approver_missing"],
    "n_pay_months": ["share_non_po", "share_pcard_expense", "manual_pay_rate", "mean_round_share", "share_tov",
                     "share_paying_mkt_differs", "share_commission_cash", "spike_ratio", "top3_month_share",
                     "cv_monthly_spend"],
    "n_inst_links": ["share_links_public"],
    "n_addresses": ["share_addr_type_missing"],
}
for base, cols in undefined_if_zero.items():
    for c in cols:
        F.loc[F[base] == 0, c] = np.nan
const = F.columns[F.nunique() <= 1]
print("dropping constant features:", list(const))
F = F.drop(columns=const)

features_L1 = X.set_index("record_id").join(F).reset_index()
features_L1.to_csv("features_L1.csv", index=False)
print("features_L1:", features_L1.shape, "->", F.shape[1], "Layer 1 variables;",
      "missing cells:", int(F.isna().sum().sum()))
display(features_L1.head())

### Checking the two authority variables

`n_exceeds_authority` looked very strong, so we should check where these approvals fall:
1. financial approvals where the approver has no valid role on the approval date (vendor, case, and why: role starts later / role already ended / no record)
2. exceeding approvals by approvers with a valid role (which vendors and cases)

Results:
- All 9 "role missing" approvals come from one approver whose role row starts in 2023-02, while the approvals are from 2012–2022. All are in CASE-0006 / CASE-0007.
- `n_exceeds_authority`: 66 approvals on 31 vendors, all of them case members. Probably planted very cleanly in this data.
- `n_approver_role_missing` is basically one person. CASE-0006 and CASE-0007 share no members, so they can land in different folds and the model can learn this approver from one case and use it on the other. That is relational information (two cases share an approver).

→ ablations without each variable and a conservative baseline without both.

In [ ]:
case_of = {}
for c in labels.itertuples():
    for m in c.members:
        case_of.setdefault(m, []).append(c.case_id)
fin_ = fin.copy()
fin_["role_valid_from"] = fin_.approver_employee_id.map(_rl.valid_from)
fin_["role_valid_to"] = fin_.approver_employee_id.map(_rl.valid_to)
fin_["role"] = fin_.approver_employee_id.map(_rl.role)
fin_["y"] = fin_.record_id.isin(set(case_of)).astype(int)
fin_["case_id"] = fin_.record_id.map(lambda r: "|".join(case_of.get(r, [])))

miss = fin_[fin_.role_missing].copy()
miss["approver_status"] = np.select(
    [~miss.approver_employee_id.isin(_rl.index),
     miss.approval_date < miss.role_valid_from,
     miss.approval_date > miss.role_valid_to.fillna(SNAPSHOT)],
    ["no record at all", "role starts later", "role already ended"], default="?")
cols = ["record_id", "y", "case_id", "engagement_id", "approver_employee_id", "approval_date", "role",
        "role_valid_from", "role_valid_to", "value_band", "required_level", "approver_status"]
print(f"(1) financial approvals with no valid role on the approval date: {len(miss)} "
      f"(approvers: {miss.approver_employee_id.nunique()}, vendors: {miss.record_id.nunique()}, "
      f"case members: {miss.drop_duplicates('record_id').y.sum()})")
display(miss[cols].sort_values(["case_id", "approval_date"]).reset_index(drop=True))
miss[cols].to_csv(f"{OUT}/L1_diag_role_missing_approvals.csv", index=False)
print("non-null approvers that never appear in employee_roles:",
      int((~fin_.approver_employee_id.dropna().isin(_rl.index)).sum()))

exc = fin_[fin_.exceeds_authority]
print(f"\n(2) exceeding approvals with a valid role: {len(exc)} approvals, {exc.record_id.nunique()} vendors, "
      f"of which case members: {exc.drop_duplicates('record_id').y.sum()}")
display(exc.drop_duplicates("record_id").groupby("case_id").record_id.count().rename("n_vendors").to_frame())

## y = 0 vs y = 1
Group means of each variable (NaN ignored), sorted by standardized mean difference. Just a first look, not used for selection.

In [ ]:
Fv = features_L1.drop(columns=["record_id", "group_id"])
m = Fv.groupby("y").mean().T
sd = Fv.drop(columns="y").std()
m["SMD"] = (m[1] - m[0]) / sd.replace(0, np.nan)
m = m.rename(columns={0: "mean | y=0", 1: "mean | y=1"}).sort_values("SMD", key=np.abs, ascending=False)
display(m.head(25).round(3))

# Supervised Learning

## Setup

`Prep()`: winsorize at 1% / 99% (variables with > 10 distinct values), then median imputation. First step of every pipeline.

Variable selection: four penalized models, keep a variable with ≥ 3 of 4 votes.
- lasso logistic
- elastic net logistic
- adaptive lasso, weights $w_j = 1/|\hat\beta^{ridge}_j|$
- L1-SVM

C is tuned by grouped inner CV on PR-AUC with the 1-SE rule (sparser model).

Models:
1. linear regression (linear probability model)
2. logistic regression, `class_weight="balanced"`
3. decision tree, `max_depth` from 2–6 chosen by inner grouped CV
4. random forest, 500 trees
5. gradient boosting (`HistGradientBoosting`, depth 3)

All training goes through `fit_model()`, both for the single split and for the 5-seed evaluation.

In [ ]:
%matplotlib inline
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
warnings.filterwarnings("ignore")
SEED = 42
OUT = "results"
os.makedirs(OUT, exist_ok=True)
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import IsolationForest
from sklearn.inspection import permutation_importance
from joblib import Parallel, delayed


class Winsorizer(BaseEstimator, TransformerMixin):
    """clip at 1% / 99% quantiles, only variables with > 10 distinct values"""
    def __init__(self, lower=0.01, upper=0.99, min_unique=10):
        self.lower, self.upper, self.min_unique = lower, upper, min_unique

    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        self.on_ = np.array([np.unique(c[~np.isnan(c)]).size > self.min_unique for c in X.T])
        return self

    def transform(self, X):
        X = np.array(X, float, copy=True)
        X[:, self.on_] = np.clip(X[:, self.on_], self.lo_[self.on_], self.hi_[self.on_])
        return X


def Prep():
    return make_pipeline(Winsorizer(), SimpleImputer(strategy="median", keep_empty_features=True))


df = features_L1.copy()
FEATS = [c for c in df.columns if c not in ("record_id", "y", "group_id")]
X_all, y_all, g_all = df[FEATS].values.astype(float), df.y.values, df.group_id.values
C_GRID = np.logspace(-3, 1, 12)

# helpers
def inner_splits(y, g, k=4, seed=SEED):
    """grouped, stratified inner CV splits"""
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
        np.zeros(len(y)), y, g))


def cv_score(make_model, X, y, splits):
    """mean PR-AUC over the splits"""
    s = []
    for tr, va in splits:
        if y[va].sum() == 0 or y[tr].sum() == 0:
            continue
        m = make_model().fit(X[tr], y[tr])
        sc = m.decision_function(X[va]) if hasattr(m, "decision_function") else m.predict_proba(X[va])[:, 1]
        s.append(average_precision_score(y[va], sc))
    return np.mean(s) if s else -np.inf


def tune_and_fit(make_model_C, X, y, splits, grid=C_GRID, one_se=True):
    """pick C by inner-CV PR-AUC; one_se: smallest C within 1 SE of the best"""
    res = []
    for C in grid:
        per = []
        for tr, va in splits:
            if y[va].sum() == 0:
                continue
            m = make_model_C(C).fit(X[tr], y[tr])
            per.append(average_precision_score(y[va], m.decision_function(X[va])))
        res.append((C, np.mean(per), np.std(per) / np.sqrt(len(per))))
    res = pd.DataFrame(res, columns=["C", "mean", "se"])
    best = res.loc[res["mean"].idxmax()]
    C_star = res.loc[res["mean"] >= best["mean"] - best["se"], "C"].min() if one_se else best["C"]
    return make_model_C(C_star).fit(X, y), C_star, res


def nonzero(coef, names, tol=1e-8):
    return [n for n, b in zip(names, np.ravel(coef)) if abs(b) > tol]


# variable selection: 4 penalized models, >= 3/4 vote
def select_variables(Xtr, ytr, gtr, names, verbose=True):
    Z = make_pipeline(Prep(), StandardScaler()).fit_transform(Xtr)
    sp = inner_splits(ytr, gtr)
    out, info = {}, {}

    # lasso
    mk = lambda C: LogisticRegression(penalty="l1", C=C, solver="liblinear",
                                      class_weight="balanced", max_iter=5000)
    m, C, _ = tune_and_fit(mk, Z, ytr, sp)
    out["lasso_logit"] = nonzero(m.coef_, names); info["lasso_logit"] = f"C={C:.4g}"

    # elastic net, tune C and l1_ratio
    best = None
    for l1r in [0.2, 0.5, 0.8]:
        mk = lambda C, l1r=l1r: LogisticRegression(penalty="elasticnet", l1_ratio=l1r, C=C,
                                                   solver="saga", class_weight="balanced",
                                                   max_iter=3000, tol=1e-3)
        m, C, res = tune_and_fit(mk, Z, ytr, sp)
        score = res["mean"].max()
        if best is None or score > best[0]:
            best = (score, m, C, l1r)
    out["elastic_net_logit"] = nonzero(best[1].coef_, names)
    info["elastic_net_logit"] = f"C={best[2]:.4g}, l1_ratio={best[3]}"

    # adaptive lasso = lasso on Z_j * |beta_ridge_j|
    ridge = LogisticRegression(penalty="l2", C=1.0, class_weight="balanced", max_iter=5000).fit(Z, ytr)
    scale = np.abs(ridge.coef_.ravel()) ** 1.0 + 1e-6
    Zs = Z * scale
    mk = lambda C: LogisticRegression(penalty="l1", C=C, solver="liblinear",
                                      class_weight="balanced", max_iter=5000)
    m, C, _ = tune_and_fit(mk, Zs, ytr, sp)
    out["adaptive_lasso_logit"] = nonzero(m.coef_, names); info["adaptive_lasso_logit"] = f"C={C:.4g}"

    # L1 SVM
    mk = lambda C: LinearSVC(penalty="l1", loss="squared_hinge", dual=False, C=C,
                             class_weight="balanced", max_iter=20000)
    m, C, _ = tune_and_fit(mk, Z, ytr, sp, grid=np.logspace(-4, 0, 15))
    out["l1_svm"] = nonzero(m.coef_, names); info["l1_svm"] = f"C={C:.4g}"

    sets = [set(v) for v in out.values()]
    inter = sorted(set.intersection(*sets))
    votes = pd.Series({n: sum(n in s for s in sets) for n in names})
    vote3 = sorted(votes[votes >= 3].index)
    if verbose:
        for k, v in out.items():
            print(f"  {k:22s} [{info[k]}] -> {len(v):3d} variables")
        print(f"  INTERSECTION of 4 -> {len(inter)} variables; >=3-of-4 vote -> {len(vote3)}")
    return out, inter, vote3, votes, info


# models 1-5
MODELS = ["1_linear_regression", "2_logistic_regression", "3_decision_tree",
          "4_random_forest", "5_gradient_boosting"]


def make_model(name, depth=None):
    if name == "1_linear_regression":
        return make_pipeline(Prep(), StandardScaler(), LinearRegression())
    if name == "2_logistic_regression":
        return make_pipeline(Prep(), StandardScaler(),
                             LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000))
    if name == "3_decision_tree":
        return make_pipeline(Prep(), DecisionTreeClassifier(max_depth=depth, min_samples_leaf=5,
                                                            class_weight="balanced", random_state=SEED))
    if name == "4_random_forest":
        return make_pipeline(Prep(), RandomForestClassifier(
            n_estimators=500, min_samples_leaf=3, max_features="sqrt",
            class_weight="balanced_subsample", random_state=SEED, n_jobs=1))
    if name == "5_gradient_boosting":
        return make_pipeline(Prep(), HistGradientBoostingClassifier(
            max_depth=3, learning_rate=0.05, max_iter=300, l2_regularization=1.0,
            class_weight="balanced", random_state=SEED))
    raise ValueError(name)


def get_score(model, X):
    # P(y=1), y_hat for linear regression
    return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.predict(X)


def fit_model(name, X, y, g):
    if name != "3_decision_tree":
        return make_model(name).fit(X, y)
    # tree: pick max_depth by inner CV
    sp = inner_splits(y, g)
    cv = {d: np.mean([average_precision_score(y[va], get_score(make_model(name, d).fit(X[tr], y[tr]), X[va]))
                      for tr, va in sp]) for d in [2, 3, 4, 5, 6]}
    best = max(cv, key=cv.get)
    m = make_model(name, best).fit(X, y)
    m.best_depth_, m.depth_cv_ = best, cv
    return m


print("p =", len(FEATS), "Layer 1 variables | missing cells:", int(np.isnan(X_all).sum()))

## Data split + variable selection

Test part = first fold of a grouped, stratified 5-fold split (201 records, 6 cases). Variable selection on the training part only.

The next sections use this split to look at each model. The selected set changes a lot between splits, so variable importance comes from the selection frequency over the 25 folds of the 5-seed evaluation, which also gives the main results.

In [ ]:
tr_idx, te_idx = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_all, y_all, g_all))
Xtr, ytr, gtr = X_all[tr_idx], y_all[tr_idx], g_all[tr_idx]
Xte, yte = X_all[te_idx], y_all[te_idx]
assert not set(gtr) & set(g_all[te_idx]), "group leakage!"
print(f"train n={len(tr_idx)} (pos={ytr.sum()}, cases={len(set(gtr[ytr == 1]))}) | "
      f"test n={len(te_idx)} (pos={yte.sum()}, cases={len(set(g_all[te_idx][yte == 1]))})\n")

sel, inter, SELECTED, votes, info = select_variables(Xtr, ytr, gtr, FEATS)
tab = pd.DataFrame({k: [f in v for f in FEATS] for k, v in sel.items()}, index=FEATS)
tab["votes"] = votes
tab = tab[tab.votes > 0].sort_values("votes", ascending=False)
tab.to_csv(f"{OUT}/L1_selection_votes.csv")
pd.Series(SELECTED, name="variable").to_csv(f"{OUT}/L1_selected_variables.csv", index=False)
print(f"\n>= 3/4 vote set: {len(SELECTED)} variables (intersection of all 4: {len(inter)})")
display(tab)

## Metrics for the single split

**`AP_case`** (main metric). Plain AP counts records, so a 13-member case counts 13× as much as a 1-member case. Evaluation is per case, so positives are reweighted:
1. weight of a positive = sum over its cases of 1 / case size (two records are in two cases); negatives = 1
2. rescale the positive weights to sum to the number of positives
3. `average_precision_score(y, score, sample_weight=w)`

The total positive weight is unchanged, so random ranking still gives ≈ 0.16, same as plain AP.

Also: `AP`, ROC-AUC, precision@k (k = number of positives in the test part), recall@2k, and precision / recall / F1 at a threshold chosen on out-of-fold predictions within the training part.

In [ ]:
# AP_case weights
_raw_w = {}
for c in labels.itertuples():
    for m in set(c.members):
        _raw_w[m] = _raw_w.get(m, 0.0) + 1.0 / len(set(c.members))


def case_weights(idx=None):
    idx = np.arange(len(df)) if idx is None else np.asarray(idx)
    y_ = y_all[idx]
    w = np.ones(len(idx))
    raw = np.array([_raw_w.get(r, 0.0) for r in df.record_id.values[idx]])
    pos = y_ == 1
    if pos.any():
        w[pos] = raw[pos] * pos.sum() / raw[pos].sum()
    return w


W_all = case_weights()
print(f"AP_case weights: positive weights sum = {W_all[y_all == 1].sum():.1f} (= #positives {int(y_all.sum())}); "
      f"positive weight range {W_all[y_all == 1].min():.2f}–{W_all[y_all == 1].max():.2f}; negatives = 1")

SEL_IDX = [FEATS.index(v) for v in SELECTED]
Xtr_s, Xte_s = Xtr[:, SEL_IDX], Xte[:, SEL_IDX]
RESULTS, TEST_SCORES, FITTED = [], {}, {}
_rng = np.random.default_rng(SEED)


def rank_metrics(y, s, thr=None, w=None):
    order = np.argsort(-(s + 1e-9 * _rng.random(len(s))))      # random tie-breaking
    k = int(y.sum())
    out = dict(AP_case=average_precision_score(y, s, sample_weight=w) if w is not None else np.nan,
               AP=average_precision_score(y, s), roc_auc=roc_auc_score(y, s),
               precision_at_k=y[order[:k]].mean(), recall_at_2k=y[order[:2 * k]].sum() / k)
    if thr is not None:
        pred = (s >= thr).astype(int)
        out.update(precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred),
                   f1=f1_score(y, pred))
    return out


def oof_threshold(name, X, y, g):
    oof = np.zeros(len(y))
    for tr, va in inner_splits(y, g):
        oof[va] = get_score(fit_model(name, X[tr], y[tr], g[tr]), X[va])
    p, r, t = precision_recall_curve(y, oof)
    f1 = 2 * p * r / np.maximum(p + r, 1e-12)
    return t[np.argmax(f1[:-1])]


def run_model(name):
    m = fit_model(name, Xtr_s, ytr, gtr)
    s = get_score(m, Xte_s)
    thr = oof_threshold(name, Xtr_s, ytr, gtr)
    RESULTS.append(dict(model=name, **rank_metrics(yte, s, thr, w=case_weights(te_idx)), threshold=thr))
    TEST_SCORES[name], FITTED[name] = s, m
    print(f"{name}: {len(SELECTED)} variables | TEST (4:1) | random-guess AP_case = AP = {yte.mean():.3f}")
    display(pd.DataFrame([RESULTS[-1]]).set_index("model").round(3))
    return m


print(f"{len(SELECTED)} selected variables | train {Xtr_s.shape} | test {Xte_s.shape}")

## Model 1: linear regression
OLS on y, rank by $\hat y$. $\hat y$ can fall outside [0, 1] and there are no thresholds or interactions. Used as a reference.

In [ ]:
model_lin = run_model("1_linear_regression")
yhat = model_lin.predict(Xte_s)
print(f"y_hat on TEST: min={yhat.min():.2f}, max={yhat.max():.2f}, "
      f"share outside [0,1] = {np.mean((yhat < 0) | (yhat > 1)):.1%}")
coef_lin = pd.Series(model_lin[-1].coef_, index=SELECTED).sort_values()
plt.figure(figsize=(7, 0.3 * len(coef_lin) + 1))
plt.barh(coef_lin.index, coef_lin.values, color=np.where(coef_lin > 0, "#C44E52", "#4C72B0"))
plt.axvline(0, c="k", lw=0.8); plt.title("Model 1 — linear regression coefficients (standardized x)")
plt.tight_layout(); plt.show()

## Model 2: logistic regression
$\log\frac{P}{1-P}=\beta_0+\beta^\top x_i$, $e^{\beta_j}$ = odds ratio per SD. `class_weight="balanced"` gives each positive ~5.3× the weight of a negative. With correlated variables (e.g. `onboard_year` and `tenure_years`) single coefficient signs can be misleading.

In [ ]:
model_logit = run_model("2_logistic_regression")
coef_logit = pd.DataFrame({"coef": model_logit[-1].coef_.ravel()}, index=SELECTED)
coef_logit["odds_ratio"] = np.exp(coef_logit.coef)
display(coef_logit.sort_values("coef", key=np.abs, ascending=False).round(3))

## Model 3: decision tree
Handles thresholds and interactions and can be plotted. Unstable, and the scores take only a few distinct values, so lots of ties (broken at random).

In [ ]:
model_tree = run_model("3_decision_tree")
print("inner-CV PR-AUC by max_depth:", {k: round(v, 3) for k, v in model_tree.depth_cv_.items()},
      "-> chosen", model_tree.best_depth_)
plt.figure(figsize=(22, 9))
plot_tree(model_tree[-1], feature_names=SELECTED, class_names=["0", "1"], filled=True,
          max_depth=3, fontsize=8, impurity=False, proportion=True)
plt.title(f"Model 3 — decision tree (top 3 levels, max_depth={model_tree.best_depth_})"); plt.show()

## Model 4: random forest
Interpreted with permutation importance on the test part (drop in AP when one variable is shuffled).

In [ ]:
model_rf = run_model("4_random_forest")
pi = permutation_importance(model_rf, Xte_s, yte, scoring="average_precision", n_repeats=20, random_state=SEED)
imp_rf = pd.Series(pi.importances_mean, index=SELECTED).sort_values()
plt.figure(figsize=(7, 0.3 * len(imp_rf) + 1))
plt.barh(imp_rf.index, imp_rf.values, xerr=pi.importances_std[np.argsort(pi.importances_mean)], color="#55A868")
plt.axvline(0, c="k", lw=0.8); plt.title("Model 4 — permutation importance on TEST (drop in PR-AUC)")
plt.tight_layout(); plt.show()

## Model 5: gradient boosting
`HistGradientBoosting`: depth 3, learning rate 0.05, 300 iterations.

In [ ]:
model_gb = run_model("5_gradient_boosting")

## The five models on the same split
Only 201 test records and 6 cases, so these numbers are noisy.

In [ ]:
res = pd.DataFrame(RESULTS).set_index("model")
res.to_csv(f"{OUT}/L1_models_test_metrics.csv")
display(res[["AP_case", "AP", "roc_auc", "precision_at_k", "recall_at_2k", "precision", "recall", "f1"]].round(3))
plt.figure(figsize=(7, 4.5))
for name, s in TEST_SCORES.items():
    p, r_, _ = precision_recall_curve(yte, s)
    plt.plot(r_, p, label=f"{name} (AP={average_precision_score(yte, s):.2f})")
plt.axhline(yte.mean(), ls="--", c="grey", lw=1, label=f"random ({yte.mean():.2f})")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("TEST PR curves (single 4:1 split)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

# Unsupervised Learning: Isolation Forest

The main dataset (~20,000 records) has no labels and may contain patterns that are not in the practice set, so we also try a label-free method.

- uses all variables (the supervised selection used the labels)
- fitted on all 1,002 records, no y
- default settings

## Preprocessing
- winsorizing + median imputation fitted on all records
- non-negative continuous variables with skew > 2 → $\log(1+x)$, so a long tail doesn't squeeze most records into a small range

Comparisons with a peer group (e.g. percentiles within the subregion) depend on other vendors, so they're left for Layer 2.

In [ ]:
def unsup_matrix(names):
    # no labels used
    Xu = pd.DataFrame(Prep().fit_transform(df[names].values.astype(float)), columns=names)
    cont_ = [c for c in names if Xu[c].nunique() > 2]
    skew_ = [c for c in cont_ if Xu[c].min() >= 0 and Xu[c].skew() > 2]
    Xu[skew_] = np.log1p(Xu[skew_])
    return Xu, cont_, skew_


XU, cont, skewed = unsup_matrix(FEATS)
print(f"{len(FEATS)} variables: {len(FEATS) - len(cont)} binary, {len(cont)} continuous, {len(skewed)} log-transformed")

## Model 6: Isolation Forest
Risk score = `-score_samples` (higher = easier to isolate). 500 trees, `max_samples=256`, averaged over 10 seeds; we also check how well the seeds agree. For the top 10 records, the variables with the largest |z| are listed.

In [ ]:
from scipy.stats import spearmanr


def iforest(Xu, seeds=range(10)):
    S = np.array([-IsolationForest(n_estimators=500, max_samples=256, contamination="auto",
                                   random_state=s).fit(Xu).score_samples(Xu) for s in seeds])
    return S.mean(0), S


IF_SCORE, S = iforest(XU.values)
rho = [spearmanr(S[i], S[j])[0] for i in range(len(S)) for j in range(i + 1, len(S))]
IF_ROWS = [dict(scope="ALL 1002", **rank_metrics(y_all, IF_SCORE, w=W_all), seed_spearman=np.mean(rho)),
           dict(scope="TEST 201", **rank_metrics(yte, IF_SCORE[te_idx], w=case_weights(te_idx)))]
display(pd.DataFrame(IF_ROWS).set_index("scope").round(3))

Z = pd.DataFrame(StandardScaler().fit_transform(XU), columns=FEATS, index=features_L1.record_id)
expl = []
for i in np.argsort(-IF_SCORE)[:10]:
    z = Z.iloc[i]
    big = z.abs().sort_values(ascending=False).head(4)
    expl.append(dict(record_id=Z.index[i], y=y_all[i], score=round(IF_SCORE[i], 3),
                     top_deviations=", ".join(f"{k} ({z[k]:+.1f}σ)" for k in big.index)))
print("\nTop-10 most anomalous records")
display(pd.DataFrame(expl))

---
# Evaluation over 5 seeds + ablations

## Red-flag rule: payments after termination

CASE-0016 is "terminated, but payments continue". `n_months_paid_after_term` captures it, but only one vendor out of 1,002 has a value > 0, so a model can't learn it (and `is_terminated = 1` pulls its score down). So we use a rule instead. It only uses the vendor's own payments and termination date.

**Rule:** flag a vendor if it is paid in month ≥ termination month + `GRACE_MONTHS` + 1 (`GRACE_MONTHS = 1`):
- the termination month itself is skipped, payments are monthly totals
- one grace month for the final invoice

The CASE-0016 vendor was terminated on 2025-01-25 and has one paid month after that, 2025-03 (`why_flagged` says "3 more month(s)"). With 2 grace months it would not be flagged. On the main data we need to check how many vendors it flags; some may be legitimate (e.g. transitional services around the consumer healthcare demerger in `business_events`).

**Ranking:** rank percentile of the score + 2 for flagged vendors, i.e. flagged vendors first, then everyone else by score. Results are reported with and without the rule.

In [ ]:
from scipy.stats import rankdata

GRACE_MONTHS = 1
_term = tp.set_index("record_id").terminated_date
_pm = pay.groupby("record_id").month.apply(lambda s: sorted(set(s.dt.to_period("M"))))


def last_paid_offset_after_termination(r):
    # months from termination month to last paid month (0 if none)
    t = _term.get(r)
    if pd.isna(t):
        return 0
    offs = [(m - t.to_period("M")).n for m in _pm.get(r, [])]
    offs = [o for o in offs if o > 0]
    return max(offs) if offs else 0


rid_all = df.record_id.values
PAY_OFFSET = pd.Series({r: last_paid_offset_after_termination(r) for r in rid_all})
case_of_all = {m: c.case_id for c in labels.itertuples() for m in c.members}
print("vendors flagged by grace period (payment in month >= termination month + grace + 1):")
display(pd.DataFrame([dict(grace_months=gm, rule=f"paid in month >= termination month + {gm + 1}",
                           n_flagged=int((PAY_OFFSET >= gm + 1).sum()),
                           flagged=", ".join(f"{r} ({case_of_all.get(r, 'not a case member')})"
                                             for r in PAY_OFFSET[PAY_OFFSET >= gm + 1].index))
                      for gm in [0, 1, 2]]))
RED_FLAG = (PAY_OFFSET.reindex(rid_all).values >= GRACE_MONTHS + 1)
print(f"GRACE_MONTHS = {GRACE_MONTHS} -> {RED_FLAG.sum()} red-flag vendor(s)")


def apply_rule(s):
    # flagged vendors on top
    return rankdata(s) / len(s) + 2.0 * RED_FLAG

## Evaluation setup

Every one of the 1,002 records gets a score from a model that didn't see its label:
- **Models 1–5:** out-of-fold predictions from grouped 5-fold CV. In each fold, variable selection is redone on the other 4 folds, the model is refit (the tree re-picks its depth), then it scores the held-out fold.
- **Isolation Forest:** no labels, scores all records directly.

Repeated for 5 seeds (5 different fold splits), reported as mean [min–max].

Metrics:
- `AP_case` (main) and `AP`, as in the single split
- `P@k`, `R@k` for k = 10, 25, 50, 100 (record-level)
- `cases_half@k`: number of the 33 cases with at least half of their members in the top k (k = 50, 100, 159; 159 = number of positives). Closest to the client's member-overlap matching.
- case recall by typology

Each model is scored with and without the red-flag rule. `run_version(names, tag)` runs 5 seeds × 5 folds for one variable set (full version and ablations).

In [ ]:
rid = df.record_id.values
case_mem = {c.case_id: set(c.members) for c in labels.itertuples()}
case_typ = dict(zip(labels.case_id, labels.typology_tags))
n_pos = int(y_all.sum())
KS, KS_CASE = [10, 25, 50, 100], [50, 100, 159]
SEEDS = list(range(N_SEEDS))
ALL_MODELS = MODELS + ["6_isolation_forest"]


def _fold(Xv, names, tr, te):
    _, _, v3, _, _ = select_variables(Xv[tr], y_all[tr], g_all[tr], names, verbose=False)
    ix = [names.index(v) for v in v3]
    out = {m: get_score(fit_model(m, Xv[tr][:, ix], y_all[tr], g_all[tr]), Xv[te][:, ix]) for m in MODELS}
    return te, v3, out


def eval_scores(s, seed):
    r = np.random.default_rng(seed)
    order = np.argsort(-(s + 1e-9 * r.random(len(s))))
    d = {"AP_case": average_precision_score(y_all, s, sample_weight=W_all), "AP": average_precision_score(y_all, s)}
    for k in KS:
        d[f"P@{k}"] = y_all[order[:k]].mean()
        d[f"R@{k}"] = y_all[order[:k]].sum() / n_pos
    for k in KS_CASE:
        top = set(rid[order[:k]])
        d[f"cases_half@{k}"] = sum(len(m & top) / len(m) >= 0.5 for m in case_mem.values())
    top = set(rid[order[:n_pos]])
    per_case = {cid: float(len(m & top) / len(m) >= 0.5) for cid, m in case_mem.items()}
    return d, per_case


FROZEN = {"oof": [], "selected": []}     # filled by run_version, saved at the end


def run_version(names, tag):
    Xv = df[names].values.astype(float)
    Xu_v = unsup_matrix(names)[0].values
    rows, cases, sels = [], [], []
    for seed in SEEDS:
        folds = list(StratifiedGroupKFold(5, shuffle=True, random_state=1000 + seed).split(Xv, y_all, g_all))
        res_ = Parallel(n_jobs=N_JOBS)(delayed(_fold)(Xv, names, tr, te) for tr, te in folds)
        S = {m: np.full(len(y_all), np.nan) for m in MODELS}
        fold_id = np.full(len(y_all), -1)
        for f_, (te, v3, out) in enumerate(res_):
            sels.append(v3)
            fold_id[te] = f_
            FROZEN["selected"].append(dict(version=tag, seed=seed, fold=f_, n_vars=len(v3), variables=v3))
            for m in MODELS:
                S[m][te] = out[m]
        S["6_isolation_forest"] = iforest(Xu_v, seeds=range(seed * 10, seed * 10 + 5))[0]
        FROZEN["oof"].append(pd.DataFrame({"version": tag, "seed": seed, "record_id": rid, "y": y_all,
                                           "group_id": g_all, "fold": fold_id, "red_flag": RED_FLAG.astype(int),
                                           **{f"score_{m}": S[m] for m in ALL_MODELS}}))
        for m in ALL_MODELS:
            nv = np.mean([len(v) for v in sels[-5:]]) if m in MODELS else len(names)
            for rule in (False, True):
                d, pc = eval_scores(apply_rule(S[m]) if rule else S[m], seed)
                rows.append(dict(version=tag, rule=rule, model=m, seed=seed, n_vars_mean=nv, **d))
                cases.append(dict(version=tag, rule=rule, model=m, seed=seed, **pc))
        print(f"  [{tag}] seed {seed + 1}/{len(SEEDS)} done")
    return pd.DataFrame(rows), pd.DataFrame(cases), sels


def summarize(rows):
    cols = ["AP_case", "AP"] + [f"P@{k}" for k in KS] + [f"R@{k}" for k in KS] + [f"cases_half@{k}" for k in KS_CASE]
    g = rows.groupby(["version", "rule", "model"], sort=False)[cols]
    mean, lo, hi = g.mean(), g.min(), g.max()
    fmt = mean.copy().astype(object)
    for c in cols:
        nd = 1 if c.startswith("cases") else 3
        fmt[c] = [f"{a:.{nd}f} [{b:.{nd}f}–{e:.{nd}f}]" for a, b, e in zip(mean[c], lo[c], hi[c])]
    return mean, fmt


def typology_table(cases):
    long = cases.melt(id_vars=["version", "rule", "model", "seed"], var_name="case_id", value_name="hit")
    long = long.assign(typology=long.case_id.map(case_typ).str.split("|")).explode("typology")
    return long.groupby(["version", "rule", "model", "typology"]).hit.mean().unstack("typology")


print("random baseline: AP_case = AP ≈", round(y_all.mean(), 3), "| cases_half@159 ≈ 3 (random ranking)")

## Full version, 5 seeds

Mean [min–max] over the 5 seeds; random ranking gives `AP_case` = `AP` ≈ 0.16.
1. all metrics, no rule
2. before / after the red-flag rule
3. case recall by typology (share of cases with ≥ half of their members in the top 159)
4. selection frequency of each variable over the 25 folds

In [ ]:
import time
t0 = time.time()
rows_full, cases_full, sels_full = run_version(FEATS, "full")
print(f"time: {(time.time() - t0) / 60:.1f} min")

mean_full, fmt_full = summarize(rows_full)
print("\nFULL version, NO rule — mean [min–max] over seeds")
display(fmt_full.xs(False, level="rule").droplevel("version"))

ba = mean_full[["AP_case", "AP", "cases_half@159"]].droplevel("version").unstack("rule")
ba.columns = [f"{m} | {'with rule' if r else 'no rule'}" for m, r in ba.columns]
for m in ["AP_case", "AP", "cases_half@159"]:
    ba[f"{m} | change"] = ba[f"{m} | with rule"] - ba[f"{m} | no rule"]
print("\nBefore / after the red-flag rule (mean over seeds)")
display(ba[[c for m in ["AP_case", "AP", "cases_half@159"] for c in ba.columns if c.startswith(m + " |")]].round(3))

typ_full = typology_table(cases_full).loc[("full", False)]
print("\nCase recall by typology (share of cases with >= half members in the top 159; mean over seeds)")
n_cases_typ = pd.Series([t for v in case_typ.values() for t in v.split("|")]).value_counts()
display(typ_full.T.assign(n_cases=n_cases_typ).sort_values("n_cases", ascending=False).round(2))

stab = pd.Series([v for s in sels_full for v in s]).value_counts() / len(sels_full)
stab = stab.reindex(FEATS).fillna(0).sort_values(ascending=False)
stab.rename("selection_frequency").to_csv(f"{OUT}/L1_selection_stability.csv")
rows_full.to_csv(f"{OUT}/L1_full_seed_metrics.csv", index=False)
cases_full.to_csv(f"{OUT}/L1_full_seed_cases.csv", index=False)
top = stab[stab > 0].head(35)[::-1]
plt.figure(figsize=(7, 0.27 * len(top) + 1))
plt.barh(top.index, top.values, color="#4C72B0")
plt.axvline(0.5, ls="--", c="grey")
plt.xlabel(f"share of {len(sels_full)} folds in which the variable is in the >=3/4 vote set")
plt.title("Selection stability (full version)"); plt.tight_layout()
plt.savefig(f"{OUT}/L1_selection_stability.png", dpi=130); plt.show()

## Paired model comparison
Within one seed all models use the same folds, so we take per-seed differences (GB − LR, RF − LR, GB − RF) and count how many of the 5 are positive.

Metrics: `AP_case`, P@10 / 25 / 50 / 100, `cases_half@159`. No rule here, it shifts every model the same way.

In [ ]:
base = rows_full[~rows_full.rule]
pairs = [("5_gradient_boosting", "2_logistic_regression", "GB − LR"),
         ("4_random_forest", "2_logistic_regression", "RF − LR"),
         ("5_gradient_boosting", "4_random_forest", "GB − RF")]
out = []
for metric in ["AP_case", "P@10", "P@25", "P@50", "P@100", "cases_half@159"]:
    pv = base.pivot(index="seed", columns="model", values=metric)
    for a_, b_, lab_ in pairs:
        d_ = pv[a_] - pv[b_]
        out.append(dict(metric=metric, pair=lab_, **{f"seed {s}": round(v, 3) for s, v in d_.items()},
                        mean=round(d_.mean(), 3), n_positive=f"{int((d_ > 0).sum())} / {len(d_)}"))
paired = pd.DataFrame(out)
paired.to_csv(f"{OUT}/L1_paired_model_comparison.csv", index=False)
display(paired)

## Ablations
Drop one group of variables and rerun everything (same seeds, selection redone in every fold, Isolation Forest refit on the smaller set).

| version | removed |
|---|---|
| −T | `is_terminated`, `tenure_years`, `onboard_year` |
| −T_ext | −T + `onboarded_last_3y`, `dd_overdue` (onboarding date and `status == 'active'` again) |
| −C | the 15 employee / institution / market variables |
| −D | the 8 country / region variables |
| −exceeds_authority | `n_exceeds_authority` |
| −role_missing | `n_approver_role_missing` |
| −authority_both | both (= conservative baseline) |

Tables are without the rule, `AP_case` first.

Why T: 60% of non-case vendors are terminated vs 18% of case members, and case members were onboarded later. No member of the APPROVER, BRIDGE, DENSE_CLUSTER, SPONSOR_APPROVER_COUPLING or ownership cases is terminated, although none of these typologies is about status. Probably a side effect of how the data was generated.

In [ ]:
GROUPS = {
    "-T": ["is_terminated", "tenure_years", "onboard_year"],
    "-T_ext": ["is_terminated", "tenure_years", "onboard_year", "onboarded_last_3y", "dd_overdue"],
    "-C": ["n_exceeds_authority", "n_business_units", "n_sponsors", "n_inst_links", "n_link_serves",
           "n_inst_countries", "share_sponsor_system", "share_cross_border", "share_paying_mkt_differs",
           "n_links_foreign", "share_links_public", "n_eng_markets", "n_distinct_approvers", "n_approver_is_sponsor",
           "n_approver_role_missing"],
    "-D": ["country_cpi", "country_cpi_tier", "country_bo_access_low", "country_sanctioned",
           "region_US", "region_EUROPE", "region_JAPAN_CHINA", "region_INTL"],
    "-exceeds_authority": ["n_exceeds_authority"],
    "-role_missing": ["n_approver_role_missing"],
    "-authority_both": ["n_exceeds_authority", "n_approver_role_missing"],
}
for k, v in GROUPS.items():
    missing = [c for c in v if c not in FEATS]
    assert not missing, (k, missing)

all_rows, all_cases = [rows_full], [cases_full]
t0 = time.time()
for tag, drop in GROUPS.items():
    names = [c for c in FEATS if c not in drop]
    r_, c_, _ = run_version(names, tag)
    all_rows.append(r_); all_cases.append(c_)
print(f"time: {(time.time() - t0) / 60:.1f} min")
abl_rows, abl_cases = pd.concat(all_rows), pd.concat(all_cases)
abl_rows.to_csv(f"{OUT}/L1_ablation_seed_metrics.csv", index=False)
abl_cases.to_csv(f"{OUT}/L1_ablation_seed_cases.csv", index=False)

abl_mean, abl_fmt = summarize(abl_rows)
abl_mean, abl_fmt = abl_mean.xs(False, level="rule"), abl_fmt.xs(False, level="rule")
order_v = ["full"] + list(GROUPS)
for c, lab_ in [("AP_case", "AP_case (main)"), ("AP", "AP (record-level)"),
                ("cases_half@159", "cases with >= half members in the top 159")]:
    print(f"\nAblation — {lab_}, mean [min–max] over seeds")
    display(abl_fmt[c].unstack("version")[order_v])
delta = abl_mean[["AP_case", "AP", "cases_half@159"]].unstack("version")
print("\nChange vs full version (mean over seeds)")
for c in ["AP_case", "AP", "cases_half@159"]:
    d_ = delta[c][order_v].sub(delta[c]["full"], axis=0).drop(columns="full")
    print(c); display(d_.round(3))

typ_abl = typology_table(abl_cases).xs(False, level="rule")
typ_abl.to_csv(f"{OUT}/L1_ablation_by_typology.csv")
fig, axes = plt.subplots(2, 3, figsize=(24, 11))
for ax, m in zip(axes.ravel(), ALL_MODELS):
    t = typ_abl.xs(m, level="model").reindex(order_v).T
    im = ax.imshow(t.values, aspect="auto", cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(order_v))); ax.set_xticklabels(order_v, rotation=30, ha="right", fontsize=8)
    ax.set_yticks(range(len(t.index))); ax.set_yticklabels(t.index, fontsize=7)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.1f}", ha="center", va="center", fontsize=6.5)
    ax.set_title(m, fontsize=10)
fig.suptitle("Case recall by typology (>= half members in top 159), mean over 5 seeds", fontsize=12)
plt.tight_layout(); plt.savefig(f"{OUT}/L1_ablation_typology.png", dpi=130); plt.show()

## Two Layer 1 baselines
- **full**: all Layer 1 variables
- **conservative**: without `n_exceeds_authority` and `n_approver_role_missing` (see the authority check), probably closer to the main data

Layer 2 is compared against both. Shown without / with the rule, plus the paired comparison on the conservative baseline.

In [ ]:
KEYS = ["AP_case", "AP", "P@10", "P@25", "P@50", "P@100", "cases_half@159"]
bl_rows = abl_rows[abl_rows.version.isin(["full", "-authority_both"])].replace({"version": {"full": "full", "-authority_both": "conservative"}})
_, bl_fmt = summarize(bl_rows)
for rule in (False, True):
    t = bl_fmt[KEYS].xs(rule, level="rule").unstack("version")
    t.columns = [f"{m} | {v}" for m, v in t.columns]
    print(f"\nLayer 1 baselines — {'WITH' if rule else 'WITHOUT'} the red-flag rule, mean [min–max] over seeds")
    display(t[[f"{m} | {v}" for m in KEYS for v in ["full", "conservative"]]])
bl_rows.to_csv(f"{OUT}/L1_baselines_seed_metrics.csv", index=False)

# paired comparison, conservative baseline
cons = abl_rows[(abl_rows.version == "-authority_both") & (~abl_rows.rule)]
out = []
for metric in ["AP_case", "P@25", "P@50", "P@100", "cases_half@159"]:
    pv = cons.pivot(index="seed", columns="model", values=metric)
    for a_, b_, lab_ in pairs:
        d_ = pv[a_] - pv[b_]
        out.append(dict(metric=metric, pair=lab_, mean=round(d_.mean(), 3), n_positive=f"{int((d_ > 0).sum())} / {len(d_)}"))
print("\nPaired model comparison on the CONSERVATIVE baseline")
display(pd.DataFrame(out))

## Cases Layer 1 misses
For each case: share of the 5 seeds in which at least half of its members are in the top 159, per model, with the rule (no-rule numbers next to it). Cases that every model misses in most seeds are the Layer 2 targets.

In [ ]:
pc0 = cases_full[~cases_full.rule].drop(columns=["version", "rule", "seed"]).groupby("model").mean().T
pc = cases_full[cases_full.rule].drop(columns=["version", "rule", "seed"]).groupby("model").mean().T
pc.insert(0, "typology_tags", pc.index.map(case_typ))
pc.insert(1, "n_members", pc.index.map(lambda c: len(case_mem[c])))
pc["best_supervised"] = pc[MODELS].max(axis=1)
pc["best_supervised_no_rule"] = pc0[MODELS].max(axis=1)
pc = pc.sort_values(["best_supervised", "6_isolation_forest"])
pc.to_csv(f"{OUT}/L1_case_recovery_full.csv")
display(pc.round(2))
targets = pc[(pc.best_supervised < 0.5) & (pc["6_isolation_forest"] < 0.5)]
print(f"\nWITH the red-flag rule: {len(targets)} of {len(pc)} cases are found (>= half members in top 159) in fewer "
      f"than half of the seeds by EVERY model -> Layer 2 target list:")
display(targets[["typology_tags", "n_members", "best_supervised", "best_supervised_no_rule", "6_isolation_forest"]].round(2))

## Save Layer 1 outputs
Saved to `results/layer1_frozen/` so Layer 2 can use the same seeds and folds (`fold` column in `oof_scores.csv`):
- `oof_scores.csv`: out-of-fold scores of models 1–6 per version × seed × record, with fold and red flag
- `fold_selected_variables.json`: selected variables per version × seed × fold
- `seed_metrics.csv`, `seed_cases.csv`: metrics and per-case hits
- `features_L1.csv`: the feature table
- `config.json`: seeds, folds, model settings, rule, variable lists, package versions, SHA-256 of the input files

In [ ]:
import json, hashlib, sklearn, platform
FZ = f"{OUT}/layer1_frozen"
os.makedirs(FZ, exist_ok=True)
oof_all = pd.concat(FROZEN["oof"], ignore_index=True)
oof_all.to_csv(f"{FZ}/oof_scores.csv", index=False)
with open(f"{FZ}/fold_selected_variables.json", "w") as fh:
    json.dump(FROZEN["selected"], fh, indent=1)
abl_rows.to_csv(f"{FZ}/seed_metrics.csv", index=False)
abl_cases.to_csv(f"{FZ}/seed_cases.csv", index=False)
features_L1.to_csv(f"{FZ}/features_L1.csv", index=False)


def sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


inputs = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".csv") and not f.startswith("features"))
config = dict(
    snapshot=str(SNAPSHOT.date()), seeds=SEEDS, fold_random_state="1000 + seed", n_folds=5,
    split_4to1_random_state=SEED, label="member of any labelled case", group="union of cases sharing a member",
    primary_metric="AP_case", main_model="5_gradient_boosting", sensitivity_model="4_random_forest",
    baselines={"full": "all Layer 1 variables", "conservative": GROUPS["-authority_both"]},
    red_flag_rule=dict(definition="payment in month >= termination month + GRACE_MONTHS + 1", GRACE_MONTHS=GRACE_MONTHS,
                       n_flagged=int(RED_FLAG.sum()), flagged=list(rid_all[RED_FLAG])),
    selection="Lasso / Elastic net / Adaptive lasso / L1-SVM on Prep+StandardScaler, C by grouped inner CV (PR-AUC, 1-SE), >=3/4 vote",
    preprocessing="Winsorizer(1%/99%, >10 distinct values) + median imputation, fitted on the training part",
    models={m: str(make_model(m, 4)[-1]) for m in MODELS},
    isolation_forest="n_estimators=500, max_samples=256, 5 seeds averaged per evaluation seed",
    ablation_groups=GROUPS, modelling_features=FEATS,
    versions=dict(python=platform.python_version(), sklearn=sklearn.__version__, pandas=pd.__version__, numpy=np.__version__),
    sha256={**{f: sha(os.path.join(DATA_DIR, f)) for f in inputs}, "features_L1.csv": sha(f"{FZ}/features_L1.csv")},
)
with open(f"{FZ}/config.json", "w") as fh:
    json.dump(config, fh, indent=1, default=str)
print("frozen to", FZ)
for f in sorted(os.listdir(FZ)):
    print(f"  {f:32s} {os.path.getsize(os.path.join(FZ, f)) / 1024:8.1f} KB")
print("oof rows:", len(oof_all), "| versions:", oof_all.version.unique().tolist())

## Layer 1 results

Mean [min–max] over 5 seeds, out-of-fold scores on all 1,002 records, variables re-selected in every fold. GB can move by ~0.01 between reruns (multithreading). Random ranking: `AP_case` = `AP` ≈ 0.16, P@k ≈ 0.16, about 3 cases with ≥ half of their members found.

**1. Baselines (GB)**

| | `AP_case` | `AP` | P@25 | cases (≥ half found) | with rule: `AP_case` / cases |
|---|---|---|---|---|---|
| full | 0.65 [0.61–0.72] | 0.64 | 0.94 | 21.6 [18–25] | 0.69 / 22.6 |
| conservative | 0.55 [0.50–0.60] | 0.48 | 0.72 | 19.8 [15–22] | 0.59 / 20.8 |

The two authority variables add about 0.10 `AP_case` and ~22 points of P@25, but only ~1.8 cases: they mostly push APPROVER members to the very top rather than finding extra cases.

**2. Model choice**
- GB has the highest `AP_case` in 5/5 seeds, vs LR (+0.056) and RF (+0.053) → GB is the main model.
- At the top of the list GB and LR are about tied on the full baseline (P@10–50 within ±0.04). On the conservative baseline GB is ahead (`AP_case` 5/5, P@50 +0.12 in 5/5).
- RF finds the most cases (+1.8 vs LR, 5/5), but its top of the list is worse (P@25 below LR in all 5 seeds). Partly unstable selection: in seed 0, fold 1 only 5 variables were kept and RF put 9 negatives at the very top. → RF as a sensitivity check.

**3. Ablations** (change in `AP_case` for GB; range over the 5 supervised models; change in cases)
- −T: −0.03 (−0.04 to −0.01); cases −1.0 to +0.8
- −T_ext: −0.09 (−0.09 to −0.03); cases −3.4 to −1.6 → most of the time / status signal comes back through `onboarded_last_3y` and `dd_overdue`; the whole channel is worth about 2–3 cases
- −C: −0.18 (−0.18 to −0.03); cases −4.0 to −0.2 → biggest group
- −D: +0.02 (−0.00 to +0.06); cases −1.2 to 0.0 → country barely matters
- −exceeds_authority: −0.13 (−0.13 to −0.03); cases −2.8 to +0.2
- −role_missing: +0.01 (−0.03 to +0.01); cases −0.4 to +0.6 → selected in 100% of folds, but its 9 vendors are all in CASE-0006 / 0007, whose other members `n_exceeds_authority` already catches, so the relational effect is small
- −authority_both: −0.11 (−0.11 to −0.00); cases −2.8 to +0.4

**4. Variables selected in ≥ 80% of the 25 folds**
`is_terminated`, `n_exceeds_authority`, `n_approver_role_missing` (1.00); `country_cpi_tier`, `n_sponsors` (0.96); `addr_changed_after_onboard`, `region_JAPAN_CHINA` (0.92); `share_commission_cash` (0.84); `region_INTL` (0.80). The single 4:1 split kept 12 variables.

**5. Red flag**
Flags only the CASE-0016 vendor; +0.03–0.04 `AP_case` and about 1 case. The grace length is based on this one example (see the red-flag rule).

**6. Layer 2 targets** (missed by every model in most seeds, even with the rule)
CASE-0004, CASE-0005 (SPONSOR_CONCENTRATION, 0 for every model), CASE-0017 (synchronized spend spikes), CASE-0026 (ownership + operational overlap), CASE-0033 (bridge + layered ownership). All five involve links between vendors or ownership chains.

**7. Isolation Forest**
`AP_case` 0.36 vs `AP` 0.24: it mostly finds single-member cases (LAYERED, BRIDGE, OWNERSHIP_CHANGE), vendors that look odd on their own. Different from what GB picks up; could combine the two rankings later.

**Open**
- how precise the authority variables are on the main data
- the red-flag grace period is based on one example
- how much of the T signal comes from the data generator